# Thesis runs on Kaggle

Settings: Accelerator = GPU T4 x2, Internet = ON, Persistence = Files. Add the private dataset `thesis-bundle` as input. Set MODELS below, then **Save Version -> Save & Run All (Commit)** so it keeps running with the browser closed (12 h limit). Output: `/kaggle/working/runs/*.jsonl`.

In [ ]:
MODELS = 'biomistral meditron'   # space-separated keys from config.yaml: llama3 mistral qwen gemma meditron biomistral
CONDITION = 'all'

In [ ]:
import glob, os, shutil, zipfile
# Kaggle may keep the zip or unzip it on upload; handle both
zips = glob.glob('/kaggle/input/*/thesis_bundle.zip')
if zips:
    zipfile.ZipFile(zips[0]).extractall('/kaggle/working')
else:
    src = os.path.dirname(glob.glob('/kaggle/input/**/bench.py', recursive=True)[0])
    shutil.copytree(src, '/kaggle/working/thesis', dirs_exist_ok=True)
os.chdir('/kaggle/working/thesis')
os.makedirs('runs', exist_ok=True)
# resume: reuse any runs already saved in a previous version of this notebook
for f in glob.glob('/kaggle/working/runs/*.jsonl'):
    dst = os.path.join('runs', os.path.basename(f))
    if not os.path.exists(dst): shutil.copy(f, dst)
print(sorted(os.listdir('.')))

In [ ]:
%%bash
pip -q install ollama pyyaml zstandard
mkdir -p /kaggle/working/ollama
curl -sL https://github.com/ollama/ollama/releases/download/v0.40.16/ollama-linux-amd64.tar.zst -o /tmp/ollama.tar.zst
ls -la /tmp/ollama.tar.zst
python -c "import zstandard, tarfile; tarfile.open(fileobj=zstandard.ZstdDecompressor().stream_reader(open('/tmp/ollama.tar.zst','rb')), mode='r|').extractall('/kaggle/working/ollama')"
/kaggle/working/ollama/bin/ollama --version
nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess, time, yaml, urllib.request
os.environ['PATH'] = '/kaggle/working/ollama/bin:' + os.environ['PATH']
os.environ['OLLAMA_FLASH_ATTENTION'] = '1'
os.environ['OLLAMA_SCHED_SPREAD'] = '1'          # one model per GPU
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '2'
server = subprocess.Popen(['ollama', 'serve'], stdout=open('/kaggle/working/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2); break
    except Exception:
        time.sleep(2)
else:
    raise SystemExit(open('/kaggle/working/ollama.log').read()[-2000:])
print('ollama server up')
cfg = yaml.safe_load(open('config.yaml'))
for m in MODELS.split():
    tag = cfg['models'][m]
    print('pulling', tag, flush=True)
    subprocess.run(['ollama', 'pull', tag], check=True)
print(subprocess.run(['ollama', 'list'], capture_output=True, text=True).stdout)

In [ ]:
import subprocess, sys, shutil, os, time
# run all MODELS at the same time (one process each); Ollama places each model on its own GPU
procs = {m: subprocess.Popen([sys.executable, '-u', 'run_experiment.py', '--model', m, '--condition', CONDITION],
                             stdout=open(f'/kaggle/working/{m}.log', 'w'), stderr=subprocess.STDOUT)
         for m in MODELS.split()}
os.makedirs('/kaggle/working/runs', exist_ok=True)
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    for f in os.listdir('runs'):
        shutil.copy(os.path.join('runs', f), '/kaggle/working/runs/' + f)
    for m in procs:
        lines = [l for l in open(f'/kaggle/working/{m}.log') if 'avg' in l or '[run]' in l]
        print(m, '|', lines[-1].strip() if lines else 'starting', flush=True)
for f in os.listdir('runs'):
    shutil.copy(os.path.join('runs', f), '/kaggle/working/runs/' + f)
print('exit codes', {m: p.returncode for m, p in procs.items()})
print('files:', os.listdir('/kaggle/working/runs'))

In [ ]:
!wc -l /kaggle/working/runs/*.jsonl